# Vertical slice: timings only

Throwaway. One series pushed through every stage once, to learn how long a fit
takes before the design commits to a number of fits. Delete this notebook once
the timings are recorded. It is not committed, has no tests, and adds nothing to
`src/fdh/`.

Series: Cornwall and Isles of Scilly (`E31000008`), Secondary Fires.

Two things here are placeholders and not decisions. Rainfall is joined on its
published label `weather_date` with no offset, which is not the alignment the
restricted condition needs. The covariate, rainfall accumulated over four weeks,
was chosen arbitrarily.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from statsforecast.models import ARIMA, AutoARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX

from fdh.config import load_config

PROCESSED = load_config().paths.resolved("processed_dir")
TERRITORY, INCIDENT_TYPE = "E31000008", "Secondary Fires"
ORDER = (1, 0, 1)
LEADS = 12
SEASON = 52  # weeks, for the seasonal naive


def fourier(steps: int, period: float, terms: int, prefix: str) -> pd.DataFrame:
    """Sine and cosine pairs on the positions 0 to steps - 1."""
    position = np.arange(steps)
    columns = {}
    for k in range(1, terms + 1):
        columns[f"{prefix}_sin{k}"] = np.sin(2 * np.pi * k * position / period)
        columns[f"{prefix}_cos{k}"] = np.cos(2 * np.pi * k * position / period)
    return pd.DataFrame(columns)


def timed(function):
    """Return wall-clock seconds and the result of one call."""
    start = time.perf_counter()
    result = function()
    return time.perf_counter() - start, result


def fit_statsmodels(y: np.ndarray, exog: np.ndarray):
    """Regression with ARMA(1, 1) errors. The intercept is the constant column of exog."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        return SARIMAX(y, exog=exog, order=ORDER, trend="n").fit(disp=False)


def fit_statsforecast(y: np.ndarray, exog: np.ndarray):
    """The same specification. include_mean adds the intercept, so exog has no constant."""
    return ARIMA(order=ORDER, include_mean=True).fit(y, exog)


timings = []  # one row per timed call

## 1. Weekly series

Weeks end on Sunday and only complete weeks are kept. Counts and rainfall are
summed into weeks. The covariate is rainfall over the four weeks ending with each
week. Annual Fourier terms sit on the weekly position, period 52.18, K = 4.

In [ ]:
counts = pd.read_parquet(PROCESSED / "daily_counts.parquet")
counts = counts[(counts.e_code_territory == TERRITORY) & (counts.incident_type == INCIDENT_TYPE)]
daily = counts.set_index(pd.to_datetime(counts.incident_date))["count"].astype(float).sort_index()

weather = pd.read_parquet(PROCESSED / "weather_panel.parquet")
weather = weather[weather.e_code_territory == TERRITORY]
rain = (
    weather.set_index(pd.to_datetime(weather.weather_date))["rainfall_mm"]
    .astype(float)
    .sort_index()
)

weeks = daily.resample("W-SUN").agg(["sum", "count"])
y_week = weeks.loc[weeks["count"] == 7, "sum"]
rain_4w = rain.resample("W-SUN").sum().rolling(4).sum().reindex(y_week.index)

X_week = fourier(len(y_week), 52.18, 4, "annual").assign(rain_4w=rain_4w.to_numpy())
X_week.insert(0, "const", 1.0)

print(f"daily: {len(daily):,} days, {daily.index[0]:%Y-%m-%d} to {daily.index[-1]:%Y-%m-%d}, "
      f"nulls {int(daily.isna().sum())}, days with no incident {int((daily == 0).sum()):,}")
print(f"weekly: {len(y_week)} complete weeks ending {y_week.index[0]:%Y-%m-%d} to "
      f"{y_week.index[-1]:%Y-%m-%d}, mean {y_week.mean():.1f} a week, "
      f"weeks with no incident {int((y_week == 0).sum())}")
print(f"regressors: {list(X_week.columns)}, nulls {int(X_week.isna().sum().sum())}")

## Origins

Ten origins spread evenly from 2015 to mid-2025. An origin is the first week
ending on or after its date. Its training window is every week before it and its
targets are that week and the eleven after it. The last origin has the longest
training window, so the single timed fits below use it and are the worst case.

In [ ]:
origin_dates = pd.date_range("2015-01-01", "2025-07-01", periods=10)
origins = [int(y_week.index.searchsorted(day)) for day in origin_dates]
assert origins[-1] + LEADS <= len(y_week), "the last origin has fewer than 12 weeks after it"
last = origins[-1]

y_w, X_w = y_week.to_numpy(), X_week.to_numpy()
for origin in origins:
    print(f"origin week ending {y_week.index[origin]:%Y-%m-%d}: {origin} training weeks")

## 2. One weekly fit in each library

Regression on the Fourier terms and the covariate with ARMA(1, 1) errors, order
(1, 0, 1). The specification is the same in both libraries; the estimation is
not. statsmodels maximises the exact likelihood through the Kalman filter.
statsforecast starts from conditional sum of squares and finishes with maximum
likelihood, its default.

In [ ]:
seconds, weekly_sm = timed(lambda: fit_statsmodels(y_w[:last], X_w[:last]))
timings.append({"arm": "weekly", "library": "statsmodels", "what": "one fit", "seconds": seconds,
                "observations": last, "regressors": X_w.shape[1],
                "converged": bool(weekly_sm.mle_retvals["converged"])})
print(f"statsmodels:   {seconds:.2f} s, converged {weekly_sm.mle_retvals['converged']}")

seconds, weekly_sf = timed(lambda: fit_statsforecast(y_w[:last], X_w[:last, 1:]))
timings.append({"arm": "weekly", "library": "statsforecast", "what": "one fit",
                "seconds": seconds, "observations": last, "regressors": X_w.shape[1],
                "converged": None})
print(f"statsforecast: {seconds:.2f} s")

compare = pd.DataFrame({
    "statsmodels": weekly_sm.forecast(steps=LEADS, exog=X_w[last:last + LEADS]),
    "statsforecast": weekly_sf.predict(LEADS, X=X_w[last:last + LEADS, 1:])["mean"],
    "observed": y_w[last:last + LEADS],
}, index=pd.RangeIndex(1, LEADS + 1, name="lead"))
print("\nForecasts from the two fits, to see they are the same model:")
print(compare.round(1).to_string())

## 3. One order search

statsforecast AutoARIMA on the same window with the same regressors, seasonal
terms off since the Fourier terms carry the season.

In [ ]:
seconds, searched = timed(lambda: AutoARIMA(seasonal=False).fit(y_w[:last], X_w[:last, 1:]))
p, q, _, _, _, d, _ = (int(value) for value in searched.model_["arma"])
timings.append({"arm": "weekly", "library": "statsforecast", "what": "order search",
                "seconds": seconds, "observations": last, "regressors": X_w.shape[1],
                "converged": None})
print(f"AutoARIMA: {seconds:.2f} s, order chosen ({p}, {d}, {q})")

## 4. Ten origins, statsmodels, perfect foresight

Each origin is fitted on its own training window and forecasts 12 weekly leads
with the observed future covariate, so this is the upper-bound condition. The
seasonal naive repeats the week 52 weeks earlier. Each origin's errors are scaled
by the mean absolute error of that seasonal naive inside the origin's training
window, so nothing from the targets reaches the scale.

In [ ]:
rows = []
loop_start = time.perf_counter()
for origin in origins:
    train = y_w[:origin]
    target = y_w[origin:origin + LEADS]
    fitted = fit_statsmodels(train, X_w[:origin])
    forecast = fitted.forecast(steps=LEADS, exog=X_w[origin:origin + LEADS])
    naive = y_w[origin - SEASON:origin - SEASON + LEADS]
    scale = np.abs(train[SEASON:] - train[:-SEASON]).mean()
    for lead in range(LEADS):
        rows.append({
            "origin": y_week.index[origin],
            "lead": lead + 1,
            "model_scaled": abs(target[lead] - forecast[lead]) / scale,
            "naive_scaled": abs(target[lead] - naive[lead]) / scale,
            "scale": scale,
        })
loop_seconds = time.perf_counter() - loop_start
errors = pd.DataFrame(rows)

print(f"{len(origins)} fits and forecasts in {loop_seconds:.1f} s, "
      f"{loop_seconds / len(origins):.2f} s each")
print("in-sample seasonal naive error used as the scale, by origin:")
print(errors.groupby("origin")["scale"].first().round(2).to_string())

## 5. Scaled error by lead, and the horizon

The mean across origins and an upper bound of the mean plus 1.96 standard errors.
The horizon is the largest lead at which the upper bound is below 1 at that lead
and at every lead before it.

In [ ]:
by_lead = errors.groupby("lead")
table = pd.DataFrame({
    "mase": by_lead["model_scaled"].mean(),
    "se": by_lead["model_scaled"].sem(),
    "naive_mase": by_lead["naive_scaled"].mean(),
})
table["upper"] = table["mase"] + 1.96 * table["se"]
table["below_1"] = table["upper"] < 1
horizon = int(table["below_1"].cummin().sum())

print(table[["mase", "se", "upper", "below_1", "naive_mase"]].round(3).to_string())
if horizon:
    print(f"\nHorizon: {horizon} week{'s' if horizon > 1 else ''} ({horizon * 7} days).")
else:
    print("\nHorizon: none. The upper bound is not below 1 at lead 1.")
print(f"The bound rests on {len(origins)} origins, so each standard error comes from "
      f"{len(origins)} values and the bound is wide. This horizon exercises the rule; "
      f"it is not a result.")

## 6. Daily arm, timing only

Daily counts with no transform. Fourier terms for the week (period 7, K = 3) and
the year (period 365.25, K = 4), and rainfall accumulated over 28 days. One fit in
each library on the days before the last origin.

In [ ]:
X_day = pd.concat(
    [fourier(len(daily), 7, 3, "week"), fourier(len(daily), 365.25, 4, "annual")], axis=1
).assign(rain_28d=rain.rolling(28).sum().reindex(daily.index).to_numpy())
X_day.insert(0, "const", 1.0)

first_target_day = y_week.index[last] - pd.Timedelta(days=6)
days = int((daily.index < first_target_day).sum())
y_d, X_d = daily.to_numpy()[:days], X_day.to_numpy()[:days]
print(f"{days:,} training days to {daily.index[days - 1]:%Y-%m-%d}, "
      f"{X_d.shape[1]} regressors, nulls {int(np.isnan(X_d).sum())}")

seconds, daily_sm = timed(lambda: fit_statsmodels(y_d, X_d))
timings.append({"arm": "daily", "library": "statsmodels", "what": "one fit", "seconds": seconds,
                "observations": days, "regressors": X_d.shape[1],
                "converged": bool(daily_sm.mle_retvals["converged"])})
print(f"statsmodels:   {seconds:.2f} s, converged {daily_sm.mle_retvals['converged']}")

seconds, daily_sf = timed(lambda: fit_statsforecast(y_d, X_d[:, 1:]))
timings.append({"arm": "daily", "library": "statsforecast", "what": "one fit",
                "seconds": seconds, "observations": days, "regressors": X_d.shape[1],
                "converged": None})
print(f"statsforecast: {seconds:.2f} s")

## 7. Timings, multiplied out

Fits = origins x series x 2 information conditions. Origins are 645 at a 7-day
stride and 162 at a 28-day stride. Series are 473 for every territory and 264
for a 24-service panel. Hours assume one fit after another on one core, each
taking as long as the single worst-case fit timed above.

In [ ]:
timing = pd.DataFrame(timings)
print(timing.assign(seconds=timing.seconds.round(2)).to_string(index=False))
for row in timing[timing.converged.eq(False)].itertuples():
    print(f"The {row.arm} {row.library} fit stopped at the default 50 iterations without "
          f"converging, so its time is a floor for a converged fit.")

fits = timing[timing.what == "one fit"].set_index(["arm", "library"])["seconds"]
plans = pd.DataFrame(
    [{"origins": o, "series": s, "fits": o * s * 2} for o in (645, 162) for s in (473, 264)]
)
for (arm, library), seconds in fits.items():
    plans[f"{arm} {library} (h)"] = (plans["fits"] * seconds / 3600).round(1)
print()
print(plans.to_string(index=False))